<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T06-L02-chunking-and-context-budget/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T06-L02-chunking-and-context-budget/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T06-L02-chunking-and-context-budget/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have. The cell below installs `tokenizers==0.23.2`, and does nothing
where they are already present. On Kaggle, switch Internet on in the notebook's settings
first; Kaggle allows that only for phone-verified accounts.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = [("tokenizers", "tokenizers==0.23.2")]            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T06-L02-chunking-and-context-budget/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T06-L02 · Chunking and the context budget

**You will build:** fixed-size and structure-aware chunkers, an approximate token-budget
counter, greedy budget-constrained context assembly with de-duplication, and the evidence
coverage metric — then measure, on your own numbers, when a chunking policy is losing the
answer before a model ever sees the question.

**Time:** ~55 minutes · **Runs on:** a laptop CPU, no GPU, no data or model download ·
**Prerequisites:** T00-L01-the-8gb-track, T06-L01-retrieval-from-scratch.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import re
import sys
import traceback
from typing import Callable, Mapping, NamedTuple, Sequence

import numpy as np
import tokenizers
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers

print("python", sys.version.split()[0], "· numpy", np.__version__, "· tokenizers",
      tokenizers.__version__, "· platform", sys.platform)

SEED = 20260923          # this lesson's one fixed seed: the corpus and every query derive
                          # from it, so re-running prints the same numbers on any machine.
TOKEN_BUDGET = 220        # the context budget every configuration below is measured against.
CANDIDATE_N = 10          # how many top-ranked chunks are offered to context assembly.
DEDUP_OVERLAP_THRESHOLD = 0.5   # see assemble_context, exercise 4.
CHARS_PER_TOKEN = 3.5     # see estimate_tokens, exercise 3, and claims.yaml.

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("fixed_size_chunk",),
    "exercise 2": ("structure_chunk",),
    "exercise 3": ("estimate_tokens",),
    "exercise 4": ("assemble_context",),
    "exercise 5": ("evidence_coverage",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (estimate_tokens)"; several -> "exercises 3 and 4"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    House guard (T06-L01, P03-L01): a stub you have not filled in yet simply says so. A wrong
    answer prints the check's own message and the notebook carries on, so one broken exercise
    never hides the feedback on the others. A demo names the exercises it `needs`: until each
    has passed its check, the demo says which one it is waiting for and skips. Every outcome is
    recorded in `_STATUS` for the progress board, and every failure in `_FAILED_CHECKS`, which
    ends a script run non-zero.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## 1. Why this lesson never calls a language model

"In RAG, a language model is augmented with an external knowledge base or a set of documents
that is passed into the context window. The data is retrieved at runtime when a query is
sent to the model" (see `claims.yaml`). Everything before generation — did the retrieved
text land in the window at all, and how much of the budget did it cost — is measurable
without generating a single token. That is this lesson: **evidence coverage**, the fraction
of a query's gold answer span that actually reaches the assembled context, against tokens
spent getting it there.

## 2. Given: the retriever and the evaluation harness, carried forward from T06-L01

`tokenize`, `Index`/`build_index` and `bm25_idf`/`bm25_score`/`bm25_rank` are T06-L01's BM25
retriever, unchanged, down to the variable names — this lesson chunks a document, then ranks
the CHUNKS with the exact same function that ranked whole documents before. `recall_at_k`,
`reciprocal_rank`, `bootstrap_ci` and `paired_bootstrap_diff` are its evaluation harness,
also unchanged: they measure whether a RELEVANT chunk was retrieved at all. Section 9 uses
them beside this lesson's own `evidence_coverage` to show the two questions are not the same
question.

In [ ]:
_TOKEN_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")


def tokenize(text: str) -> list[str]:
    """Lower-case `text` and split it into tokens (T06-L01, unchanged)."""
    return _TOKEN_RE.findall(text.lower())


class Index(NamedTuple):
    doc_term_counts: list[dict[str, int]]
    doc_lengths: list[int]
    avgdl: float
    df: dict[str, int]
    n_docs: int


def build_index(docs: Sequence[str]) -> Index:
    """Tokenise every document in `docs` and index it for BM25 (T06-L01, unchanged)."""
    doc_term_counts: list[dict[str, int]] = []
    doc_lengths: list[int] = []
    df: dict[str, int] = {}
    for doc in docs:
        counts: dict[str, int] = {}
        for token in tokenize(doc):
            counts[token] = counts.get(token, 0) + 1
        doc_term_counts.append(counts)
        doc_lengths.append(sum(counts.values()))
        for term in counts:
            df[term] = df.get(term, 0) + 1
    n_docs = len(docs)
    avgdl = sum(doc_lengths) / n_docs if n_docs else 0.0
    return Index(doc_term_counts, doc_lengths, avgdl, df, n_docs)


def bm25_idf(term: str, index: Index) -> float:
    """BM25's inverse-document-frequency weight for `term` (T06-L01, unchanged)."""
    n_t = index.df.get(term, 0)
    N = index.n_docs
    return math.log((N - n_t + 0.5) / (n_t + 0.5))


def bm25_score(query_tokens: Sequence[str], doc_id: int, index: Index,
               k1: float = 1.2, b: float = 0.75) -> float:
    """BM25's score for document `doc_id` against `query_tokens` (T06-L01, unchanged)."""
    query_term_counts: dict[str, int] = {}
    for token in query_tokens:
        query_term_counts[token] = query_term_counts.get(token, 0) + 1
    counts = index.doc_term_counts[doc_id]
    dl = index.doc_lengths[doc_id]
    avgdl = index.avgdl if index.avgdl else 1.0
    score = 0.0
    for term, qtf in query_term_counts.items():
        tf = counts.get(term, 0)
        if tf == 0:
            continue
        idf = bm25_idf(term, index)
        denom = k1 * ((1 - b) + b * dl / avgdl) + tf
        score += qtf * idf * tf / denom
    return score


def bm25_rank(query_tokens: Sequence[str], index: Index,
              k1: float = 1.2, b: float = 0.75) -> list[tuple[int, float]]:
    """Rank every document in `index` against `query_tokens` by BM25 score (T06-L01, unchanged)."""
    scored = [(doc_id, bm25_score(query_tokens, doc_id, index, k1, b))
              for doc_id in range(index.n_docs)]
    scored = [(doc_id, score) for doc_id, score in scored if score != 0.0]
    scored.sort(key=lambda pair: (-pair[1], pair[0]))
    return scored


def recall_at_k(ranked_ids: Sequence[int], relevant_ids: Sequence[int] | frozenset[int],
                 k: int) -> float:
    """The fraction of `relevant_ids` in the top `k` of `ranked_ids` (T06-L01, unchanged)."""
    relevant = set(relevant_ids)
    if not relevant:
        raise ValueError("recall_at_k needs at least one relevant id")
    top_k = set(ranked_ids[:k])
    return len(top_k & relevant) / len(relevant)


def reciprocal_rank(ranked_ids: Sequence[int], relevant_ids: Sequence[int] | frozenset[int]) -> float:
    """1 / (rank of the first relevant id), or 0.0 if none is present (T06-L01, unchanged)."""
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(ranked_ids, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0


def bootstrap_ci(values: Sequence[float], n_boot: int = 2000, seed: int = 0,
                  alpha: float = 0.05) -> tuple[float, float, float]:
    """A percentile bootstrap confidence interval for the mean of `values` (T06-L01, unchanged)."""
    arr = np.asarray(values, dtype=float)
    n = len(arr)
    rng = np.random.default_rng(seed)
    resample_idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = arr[resample_idx].mean(axis=1)
    lower, upper = np.percentile(boot_means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(arr.mean()), float(lower), float(upper)


def paired_bootstrap_diff(a: Sequence[float], b: Sequence[float], n_boot: int = 2000,
                           seed: int = 0, alpha: float = 0.05) -> tuple[float, float, float]:
    """A percentile bootstrap CI for the mean of `b[i]-a[i]`, paired (T06-L01, unchanged)."""
    a_arr = np.asarray(a, dtype=float)
    b_arr = np.asarray(b, dtype=float)
    n = len(a_arr)
    rng = np.random.default_rng(seed)
    resample_idx = rng.integers(0, n, size=(n_boot, n))
    diffs = b_arr[resample_idx] - a_arr[resample_idx]
    boot_means = diffs.mean(axis=1)
    lower, upper = np.percentile(boot_means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float((b_arr - a_arr).mean()), float(lower), float(upper)

## 3. The corpus: eight incident reports, one planted fact each (given)

`build_documents()` writes eight short "incident report" documents: a heading, then five
paragraphs of procedural filler drawn from a fixed pool, with ONE paragraph carrying the
fact a query needs — a lead filler sentence, the evidence itself, a trailing filler sentence.
Half the topics plant a SHORT one-sentence evidence span, the other half a LONG two-sentence
one (the cell below prints both lengths) — the "varying length" section 9 measures coverage
against. The evidence names a case code, a root cause and where it was found; the heading
deliberately does NOT repeat the code, so a query for it has one right answer, not two. The
gold span runs from the evidence's first token to its LAST token: the same units a chunk's
own span is measured in (exercise 1), so no chunk is ever scored as missing the fact merely
for want of the closing full stop, which no token owns.

Two queries per document: the case code alone, and two "cause words" that, by construction,
appear only inside that document's evidence sentence — so BM25 can always find the right
document. What differs, chunk by chunk, is whether the text it finds contains the WHOLE fact.

In [ ]:
TOPICS = [
    dict(name="hydraulics", cause_words=["cracked", "gasket"], cause_phrase="a cracked gasket",
         part="manifold assembly 12", inspect_word="pressure"),
    dict(name="networking", cause_words=["misconfigured", "route"],
         cause_phrase="a misconfigured route", part="edge router R4", inspect_word="packet"),
    dict(name="finance-ops", cause_words=["duplicate", "remittance"],
         cause_phrase="a duplicate remittance", part="the payables batch job",
         inspect_word="ledger"),
    dict(name="ceramics", cause_words=["uneven", "firing"], cause_phrase="an uneven firing curve",
         part="kiln zone 3", inspect_word="thermal"),
    dict(name="spacecraft", cause_words=["stuck", "thruster"],
         cause_phrase="a stuck thruster valve", part="the attitude control module",
         inspect_word="telemetry"),
    dict(name="gardening", cause_words=["waterlogged", "soil"], cause_phrase="waterlogged soil",
         part="bed 7", inspect_word="drainage"),
    dict(name="cryptography", cause_words=["reused", "nonce"], cause_phrase="a reused nonce",
         part="the handshake routine", inspect_word="checksum"),
    dict(name="hydrogeology", cause_words=["sediment", "intrusion"],
         cause_phrase="sediment intrusion", part="borehole 12", inspect_word="turbidity"),
]
# Every topic's cause words are unique to it, so a "cause" query cannot accidentally match
# another topic's evidence sentence. Checked once, here, rather than trusted.
_all_cause_words = [w for t in TOPICS for w in t["cause_words"]]
assert len(_all_cause_words) == len(set(_all_cause_words)), "two topics share a cause word"

GENERIC_FILLER_SENTENCES = [
    "This report was compiled by the on-call reviewer after the ticket queue flagged the case "
    "for follow-up.",
    "Standard procedure requires every incident above severity two to be documented within "
    "one business week of closure.",
    "The on-call rotation logged three related tickets in the same reporting period, none of "
    "which met the escalation threshold.",
    "A weekly summary of closed tickets is circulated to the operations distribution list "
    "every Friday afternoon.",
    "No customer-facing service level agreement was breached during the investigation window.",
    "The reviewing team confirmed that the affected system remained within its normal "
    "operating envelope throughout.",
    "Change management records show no unrelated deployment in the six hours preceding the "
    "report.",
    "A follow-up review is scheduled for the next quarterly operations meeting.",
    "The incident channel was archived once the on-call engineer marked the ticket resolved.",
    "Historical data for this reporting period is retained for eighteen months under the "
    "standard retention policy.",
    "The report references internal ticket numbers that are not reproduced here for brevity.",
    "This section intentionally summarises process rather than technical detail.",
]


class Document(NamedTuple):
    """One generated incident report and where its gold evidence sits inside `text`."""

    doc_id: int
    topic: str
    text: str
    evidence_start: int   # character offset into `text`
    evidence_end: int     # character offset into `text`, exclusive


def build_documents(seed: int = SEED) -> list[Document]:
    """Generate the eight-topic corpus described above from a numpy Generator seeded `seed`."""
    rng = np.random.default_rng(seed)
    docs: list[Document] = []
    for ti, topic in enumerate(TOPICS):
        code = f"case-{4800 + ti * 37}"
        date = f"2024-{(ti % 8) + 1:02d}-{10 + ti:02d}"
        heading = f"## Incident Report: {topic['name'].title()}"

        def pick_filler(n: int) -> list[str]:
            return rng.choice(GENERIC_FILLER_SENTENCES, size=n, replace=False).tolist()

        para0 = " ".join(pick_filler(3))
        para1 = " ".join(pick_filler(2))
        lead = pick_filler(1)[0]
        trail = pick_filler(1)[0]

        if ti % 2 == 0:
            evidence_text = (f"Root cause: {topic['cause_phrase']}, isolated in {topic['part']}, "
                             f"logged under {code} on {date}.")
        else:
            evidence_text = (
                f"Root cause: {topic['cause_phrase']}, isolated in {topic['part']}, "
                f"logged under {code} on {date}. Technicians confirmed the failure mode with a "
                f"manual {topic['inspect_word']} inspection, and the current safeguards will "
                f"not prevent a recurrence without a design change to {topic['part']}.")

        para2 = f"{lead} {evidence_text} {trail}"
        para3 = " ".join(pick_filler(2))
        para4 = " ".join(pick_filler(1))

        text = heading + "\n\n" + "\n\n".join([para0, para1, para2, para3, para4])
        evidence_start = text.index(evidence_text)
        # Token-aligned, like every chunk span: first token's start to LAST token's end, so the
        # closing full stop (which no token owns) is not part of the gold span.
        evidence_end = evidence_start + len(evidence_text.rstrip("."))
        docs.append(Document(ti, topic["name"], text, evidence_start, evidence_end))
    return docs


class RagQuery(NamedTuple):
    """One query: which document has the answer, and its gold evidence span within it."""

    qid: str
    doc_id: int
    tokens: list[str]
    evidence_start: int
    evidence_end: int


def build_queries(docs: Sequence[Document]) -> list[RagQuery]:
    """Two queries per document: the case code, and its two cause words."""
    queries: list[RagQuery] = []
    for doc in docs:
        code = f"case-{4800 + doc.doc_id * 37}"
        queries.append(RagQuery(f"{doc.doc_id}-code", doc.doc_id, tokenize(code),
                                 doc.evidence_start, doc.evidence_end))
        queries.append(RagQuery(f"{doc.doc_id}-cause", doc.doc_id,
                                 list(TOPICS[doc.doc_id]["cause_words"]),
                                 doc.evidence_start, doc.evidence_end))
    return queries


def _show_corpus_sample() -> None:
    docs = build_documents()
    queries = build_queries(docs)
    d = docs[1]
    print(f"{len(docs)} documents, {len(queries)} queries, doc lengths "
          f"{min(len(tokenize(x.text)) for x in docs)}-"
          f"{max(len(tokenize(x.text)) for x in docs)} tokens")
    spans = [(x.doc_id % 2, len(tokenize(x.text[x.evidence_start:x.evidence_end]))) for x in docs]
    short = [n for parity, n in spans if parity == 0]
    long_ = [n for parity, n in spans if parity == 1]
    print(f"evidence spans: short {min(short)}-{max(short)} tokens, "
          f"long {min(long_)}-{max(long_)} tokens\n")
    print(f"document {d.doc_id} ({d.topic}):\n{d.text}\n")
    print(f"its evidence span ({d.evidence_end - d.evidence_start} characters):\n  "
          f"{d.text[d.evidence_start:d.evidence_end]!r}")


_show_corpus_sample()

## 4. Exercise 1 — fixed-size chunking with overlap: `fixed_size_chunk`

The simplest chunking policy: cut a document into windows of `chunk_size` TOKENS, stepping
forward by `chunk_size - overlap` tokens each time, so consecutive windows share `overlap`
tokens. A token here, and in exercise 2, is a `tokenize` token (roughly a word); exercise 3
prices finished text in LLM tokens, separately. `_offset_tokens` (given) tokenises `text`
with the SAME rule as `tokenize`, but also returns each token's character offsets — what a
chunker needs to report a chunk's span in the ORIGINAL document, not just its own text.

<details><summary>💡 Hint 1 — what to think about</summary>

`_offset_tokens("a b c")` gives you each token's own `(start, end)`. A window covering
tokens `start_idx` through `end_idx - 1` has a character span running from the FIRST of
those tokens' start to the LAST of those tokens' end — not from wherever the previous window
left off. What should happen when `start + chunk_size` would run past the last token? And
what must stop the loop from producing a second window identical to a first that already
reached the end?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate `chunk_size` and `overlap` first (a non-positive size, or an overlap that is not
strictly smaller than the size, can never make forward progress). Tokenise once. Walk a
`start` index forward by `chunk_size - overlap` each time; each window's `end` index is
`start + chunk_size`, CLAMPED to the token count, and its character span is the first
token's start to the (clamped) last token's end. The moment a window's `end` reaches the
final token, record it and STOP — do not advance `start` again.
</details>

In [ ]:
class ChunkSpan(NamedTuple):
    """A chunk's own text and its character span within the document it was cut from."""

    text: str
    start: int
    end: int


class Chunk(NamedTuple):
    """A `ChunkSpan` labelled with which document it came from, for a flat multi-document index."""

    doc_id: int
    chunk_id: int
    text: str
    start: int
    end: int


def _offset_tokens(text: str) -> list[tuple[str, int, int]]:
    """Given, not graded: `(token, start, end)` for every token in `text`, same rule as `tokenize`."""
    return [(m.group(0), m.start(), m.end()) for m in _TOKEN_RE.finditer(text.lower())]


def fixed_size_chunk(text: str, chunk_size: int, overlap: int = 0) -> list[ChunkSpan]:
    """Split `text` into windows of `chunk_size` tokens, stepping by `chunk_size - overlap`.

    Requirements, each of which is graded:
      * `chunk_size` must be `> 0`; `overlap` must be `>= 0` and STRICTLY LESS than
        `chunk_size` — raise `ValueError` otherwise (equal or greater would never advance).
      * every token in `text` appears in at least one returned chunk — nothing dropped.
      * the LAST window may hold fewer than `chunk_size` tokens; it must not be followed by a
        further, redundant window that covers the same trailing tokens again.
      * a chunk's character span runs from its FIRST token's start to its LAST token's end.
      * `text` with no tokens at all returns an empty list.

    Example:
        >>> [s.text for s in fixed_size_chunk("a b c d e", chunk_size=3, overlap=1)]
        ['a b c', 'c d e']
    """
    # YOUR CODE HERE
    raise NotImplementedError


def chunk_document(doc: Document, chunk_fn: Callable[..., list[ChunkSpan]], **kwargs) -> list[Chunk]:
    """Given, not graded: apply `chunk_fn` to `doc.text` and label the results with `doc.doc_id`."""
    spans = chunk_fn(doc.text, **kwargs)
    return [Chunk(doc.doc_id, i, s.text, s.start, s.end) for i, s in enumerate(spans)]


# Public checks — run these as often as you like.
def _check_fixed_size_chunk() -> None:
    text = "a b c d e f g h"
    spans = fixed_size_chunk(text, chunk_size=3, overlap=1)
    assert [s.text for s in spans] == ["a b c", "c d e", "e f g", "g h"], (
        f"got {[s.text for s in spans]} — step forward by chunk_size - overlap = 2 tokens each "
        "window; stop once a window reaches the LAST token"
    )
    # With a 1-token overlap, consecutive windows SHARE a token, so their character spans are
    # contiguous end to end: the whole string, spaces included, must be covered with no gap.
    covered = set()
    for s in spans:
        covered |= set(range(s.start, s.end))
    assert covered == set(range(len(text))), (
        f"covered {sorted(covered)}, expected every character 0-{len(text) - 1} — with "
        "overlap=1 consecutive windows share a token, so there must be no gap between them"
    )
    assert fixed_size_chunk("", chunk_size=5) == [], "no tokens at all must return an empty list"
    solo = fixed_size_chunk("one two", chunk_size=10)
    assert len(solo) == 1 and solo[0].text == "one two", (
        "a document shorter than chunk_size must come back as ONE chunk, not zero and not padded"
    )
    for bad in (dict(chunk_size=0), dict(chunk_size=5, overlap=5), dict(chunk_size=5, overlap=6)):
        try:
            fixed_size_chunk("a b c", **bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"fixed_size_chunk(**{bad}) must raise ValueError")
    print("exercise 1 looks right — fixed_size_chunk steps by chunk_size - overlap, covers "
          "every token, and rejects a non-positive size or an overlap that would never advance")

In [ ]:
_try("exercise 1", _check_fixed_size_chunk)

In [ ]:
def _show_fixed_size_on_corpus() -> None:
    docs = build_documents()
    d = docs[1]   # a "long evidence" document
    for cs, ov in ((15, 0), (60, 0)):
        chunks = chunk_document(d, fixed_size_chunk, chunk_size=cs, overlap=ov)
        crosses = [c for c in chunks
                   if c.start < d.evidence_end and c.end > d.evidence_start
                   and not (c.start <= d.evidence_start and c.end >= d.evidence_end)]
        print(f"chunk_size={cs:3d} overlap={ov}: {len(chunks)} chunks, evidence span "
              f"({d.evidence_end - d.evidence_start} chars) split across "
              f"{len(crosses)} of them")


_try("fixed-size on corpus", _show_fixed_size_on_corpus, needs=("exercise 1",))

## 5. Exercise 2 — structure-aware chunking: `structure_chunk`

A fixed-size window has no idea a paragraph break is a natural place to cut. `_split_blocks`
(given) splits `text` on blank-line paragraph breaks into blocks, each with its own
character span, and merges a heading line (`#...`) into the SAME block as the paragraph
that follows it, so a heading is never left to close a chunk alone. Grouping those blocks
into token-budgeted chunks is graded.

<details><summary>💡 Hint 1 — what to think about</summary>

Walk `_split_blocks(text)`'s blocks in the order they come. You are always doing ONE of
three things: growing the chunk you are currently building, closing it and starting a new
one, or — for a block too big to ever fit inside `max_tokens` on its own — handling it as a
special case that does not touch the chunk you were building at all. What must happen to a
chunk you were already growing when the NEXT block would push it over budget? What must
happen to it BEFORE an oversized block is handled, so its tokens are not lost?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Track the current chunk's start offset, end offset and running token count (in `tokenize`
tokens) as you go. For each block: if the block's own count already exceeds `max_tokens`,
first CLOSE whatever chunk you were accumulating (if any — append it to the result and
reset), then split the block on its own with `fixed_size_chunk(block_text, max_tokens,
overlap=0)`, re-basing each piece's offsets by the block's own start in the document, and
move on to the next block. Otherwise: if you have no chunk open yet, open one with this block. If adding
this block would keep the running total `<= max_tokens`, extend the current chunk. If it
would not, close the current chunk and open a new one starting with this block. After the
loop, close whatever chunk is still open.
</details>

In [ ]:
def _split_blocks(text: str) -> list[ChunkSpan]:
    """Given, not graded: `text`'s non-empty paragraph blocks, each with its own char span.

    A heading line (starting with `#`) is merged into the block of the paragraph immediately
    after it.
    """
    raw_blocks: list[ChunkSpan] = []
    pos = 0
    for raw in text.split("\n\n"):
        start = text.index(raw, pos)
        end = start + len(raw)
        pos = end
        if raw.strip():
            raw_blocks.append(ChunkSpan(raw, start, end))
    merged: list[ChunkSpan] = []
    i = 0
    while i < len(raw_blocks):
        block = raw_blocks[i]
        if block.text.lstrip().startswith("#") and i + 1 < len(raw_blocks):
            nxt = raw_blocks[i + 1]
            merged.append(ChunkSpan(text[block.start:nxt.end], block.start, nxt.end))
            i += 2
        else:
            merged.append(block)
            i += 1
    return merged


def structure_chunk(text: str, max_tokens: int) -> list[ChunkSpan]:
    """Group `_split_blocks(text)`'s blocks into chunks of at most `max_tokens` tokens each.

    Requirements, each of which is graded:
      * `max_tokens` must be `> 0` — raise `ValueError` otherwise, whatever `text` holds.
      * a block's token count is `len(tokenize(block_text))` — the same tokens
        `fixed_size_chunk` windows over, NOT `estimate_tokens` (that prices a finished chunk
        for the context budget in exercise 4).
      * blocks are appended to the CURRENT chunk while doing so keeps its token count `<=
        max_tokens`; the moment the next block would exceed it, the current chunk is CLOSED
        and a new one starts with that block.
      * a single block that ALONE exceeds `max_tokens` cannot be grouped with anything: the
        chunk you were already accumulating (if any) must be closed FIRST, then the oversized
        block is split on its own with `fixed_size_chunk(block_text, max_tokens, overlap=0)`,
        each piece's offsets re-based onto the block's own position in `text` — never raise,
        and never leave the block as one silently over-budget chunk.
      * chunk spans are returned in document order, and every chunk's `text` is exactly
        `text[start:end]`; every token any block contains ends up in exactly one returned
        chunk (none dropped, none duplicated).

    Example:
        >>> text = "# H\\n\\nshort para one.\\n\\nshort para two."
        >>> [s.text for s in structure_chunk(text, max_tokens=100)]
        ['# H\\n\\nshort para one.\\n\\nshort para two.']
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_structure_chunk() -> None:
    text = "# Heading\n\nfirst paragraph here.\n\nsecond paragraph here.\n\nthird one too."
    chunks = structure_chunk(text, max_tokens=100)
    assert len(chunks) == 1 and "Heading" in chunks[0].text and "third" in chunks[0].text, (
        "a generous max_tokens should keep the whole document — heading and all — as ONE chunk"
    )
    tight = structure_chunk(text, max_tokens=6)
    assert len(tight) >= 2, (
        f"got {len(tight)} chunk(s) — a tight max_tokens must close a chunk once the NEXT "
        "block would push it over budget, not keep growing it"
    )
    assert "Heading" in tight[0].text and "first" in tight[0].text, (
        "the heading must be merged into the SAME chunk as the paragraph after it, never left "
        "to close a chunk alone"
    )
    total_tokens = sum(len(tokenize(c.text)) for c in tight)
    assert total_tokens == len(tokenize(text)), (
        "every token in the document must appear in exactly one chunk — none dropped, none "
        "duplicated"
    )
    snug = structure_chunk("one two.\n\nthree.", max_tokens=3)
    assert len(snug) == 1, (
        f"got {len(snug)} chunks — blocks of 2 and 1 `tokenize` tokens fill a max_tokens of 3 "
        "exactly, and the budget is inclusive (<=). Count a block with tokenize, not "
        "estimate_tokens"
    )
    huge_para = "word " * 20
    oversized = structure_chunk(f"# H\n\n{huge_para.strip()}", max_tokens=5)
    assert len(oversized) > 1 and all(len(tokenize(c.text)) <= 5 for c in oversized), (
        "a single block that alone exceeds max_tokens must be split with fixed_size_chunk, "
        "not raise and not stay as one over-budget chunk"
    )
    # A chunk already being accumulated must be FLUSHED before the fallback runs, not merged
    # into it: "lead" (small) has to close its own chunk before the huge middle paragraph is
    # split, and "tail" (small) must start a fresh chunk afterwards, not attach to a fallback
    # piece.
    three_blocks = f"# H\n\nshort lead.\n\n{huge_para.strip()}\n\nshort tail."
    mixed = structure_chunk(three_blocks, max_tokens=5)
    assert any("lead" in c.text for c in mixed) and any("tail" in c.text for c in mixed), (
        "the lead and tail paragraphs must still appear in the result, each in its own chunk"
    )
    assert not any("lead" in c.text and "word" in c.text for c in mixed), (
        "the chunk holding 'lead' must be CLOSED before the oversized paragraph's fallback "
        "pieces are added, not grown to include them"
    )
    assert sum(len(tokenize(c.text)) for c in mixed) == len(tokenize(three_blocks)), (
        "every token across all three paragraphs must still appear in exactly one chunk"
    )
    assert all(three_blocks[c.start:c.end] == c.text for c in mixed), (
        "a chunk's start/end must locate its text in the DOCUMENT — a fallback piece's offsets "
        "are relative to its block until you add the block's own start"
    )
    try:
        structure_chunk(text, max_tokens=0)
    except ValueError:
        pass
    else:
        raise AssertionError("structure_chunk(..., max_tokens=0) must raise ValueError")
    print("exercise 2 looks right — structure_chunk groups blocks under budget, keeps a "
          "heading with its paragraph, and falls back on an over-budget block")

In [ ]:
_try("exercise 2", _check_structure_chunk)

In [ ]:
def _show_structure_vs_fixed() -> None:
    docs = build_documents()
    d = docs[1]
    fixed = chunk_document(d, fixed_size_chunk, chunk_size=20, overlap=0)
    structured = chunk_document(d, structure_chunk, max_tokens=90)

    def covers_evidence(chunks: Sequence[Chunk]) -> bool:
        return any(c.start <= d.evidence_start and c.end >= d.evidence_end for c in chunks)

    print(f"fixed_size_chunk(20, 0):     {len(fixed)} chunks, one chunk fully contains the "
          f"evidence span: {covers_evidence(fixed)}")
    print(f"structure_chunk(max_tokens=90): {len(structured)} chunks, one chunk fully "
          f"contains the evidence span: {covers_evidence(structured)}")
    print("\nthe evidence lives inside ONE paragraph in this corpus (by construction) — a "
          "chunker that respects paragraph boundaries keeps it whole; one that does not may cut")


_try("structure vs fixed", _show_structure_vs_fixed, needs=("exercise 1", "exercise 2"))

## 6. Exercise 3 — the token-budget counter: `estimate_tokens`

A real subword tokenizer is the honest way to count tokens, but this lesson approximates
instead: "For Claude, a token approximately represents 3.5 English characters" (see
`claims.yaml`). Chunking and assembly decisions here do not need to be exact to one model's
vocabulary — no vocabulary this lesson could train would match the model reading the
context anyway — so a documented, fast, dependency-free approximation is used for every
GRADED budget decision below, and its real error is measured (not asserted) two cells down
against a genuine byte-pair-encoding tokenizer trained, offline, on this lesson's own corpus.

<details><summary>💡 Hint — the approach, in words</summary>

Divide the character count by `CHARS_PER_TOKEN`. A real tokenizer would never charge a
fraction of a token, and rounding down would UNDER-count (letting more text than really
fits sneak under the budget) — which direction should you round instead? And what should a
genuinely empty string cost, versus the smallest possible non-empty one?
</details>

In [ ]:
def estimate_tokens(text: str) -> int:
    """Approximate `text`'s LLM token cost as `ceil(len(text) / CHARS_PER_TOKEN)`.

    Requirements, each of which is graded:
      * an empty string costs exactly `0` tokens.
      * any NON-empty string costs at least `1` token — never `0`.
      * otherwise, the character count divided by `CHARS_PER_TOKEN`, rounded UP (not
        truncated) — a real tokenizer never charges a fractional token, and rounding down
        would under-count exactly where a budget decision is closest to the edge.

    Example:
        >>> estimate_tokens("")
        0
        >>> estimate_tokens("abcd")
        2
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_estimate_tokens() -> None:
    assert estimate_tokens("") == 0, "an empty string must cost exactly 0 tokens"
    assert estimate_tokens("a") == 1, (
        "even one character must cost at least 1 token, never 0 — a real tokenizer agrees"
    )
    # 15 is not a multiple of CHARS_PER_TOKEN, so truncating and rounding up disagree on it;
    # 14 is, so there is nothing to round and adding 1 "to be safe" over-counts it.
    for n in (15, 14):
        expected = math.ceil(n / CHARS_PER_TOKEN)
        got = estimate_tokens("a" * n)
        assert got == expected, (
            f"{n} characters at {CHARS_PER_TOKEN} chars/token should be ceil({n}/"
            f"{CHARS_PER_TOKEN}) = {expected}, got {got} — round UP rather than truncating, "
            "and never add 1 to a token count that is already whole"
        )
    longer = estimate_tokens("word " * 50)
    shorter = estimate_tokens("word " * 10)
    assert longer > shorter, "more characters must never estimate FEWER tokens"
    print("exercise 3 looks right — estimate_tokens rounds up, never returns 0 for non-empty "
          "text, and grows with length")

In [ ]:
_try("exercise 3", _check_estimate_tokens)

In [ ]:
def _train_reference_tokenizer(texts: Sequence[str], vocab_size: int = 300) -> Tokenizer:
    """Given, not graded: a small byte-pair-encoding vocabulary trained on `texts`, offline.

    Follows the same `tokenizers` recipe as P02-L05-clause-classification-abstention: a fresh
    `Tokenizer(models.BPE(...))`, lower-cased, split on whitespace, trained ONLY on the texts
    passed in — no network, no pretrained checkpoint.
    """
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.Lowercase()
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=2,
                                  special_tokens=["[UNK]"], show_progress=False)
    tok.train_from_iterator(list(texts), trainer)
    return tok


def _show_approximation_error() -> None:
    docs = build_documents()
    tok = _train_reference_tokenizer([d.text for d in docs])
    errors = []
    for d in docs:
        real = len(tok.encode(d.text).ids)
        approx = estimate_tokens(d.text)
        err = abs(approx - real) / real
        errors.append(err)
        print(f"{d.topic:14s} real={real:4d} tokens   approx={approx:4d} tokens   "
              f"error={err * 100:4.1f}%")
    worst = int(np.argmax(errors))
    print(f"\nmean absolute error over {len(docs)} documents: {np.mean(errors) * 100:.1f}%; "
          f"worst document: {docs[worst].topic}, {errors[worst] * 100:.1f}%")
    print(f"a budget sized with this estimate needs at least {errors[worst] * 100:.1f}% headroom "
          "— and that is against a vocabulary trained on this corpus, not the model's own.")


_try("approximation error demo", _show_approximation_error, needs=("exercise 3",))

## 7. Exercise 4 — assembling a context under budget: `assemble_context`

Given candidate chunks best-first, spend the token budget greedily: take a chunk if it fits,
SKIP one that does not (a smaller chunk further down the ranking may still fit), and never
spend budget twice on text you already have. `_covered_fraction` (given) measures how much
of a candidate's own span is already covered by chunks already picked from the SAME document.

<details><summary>💡 Hint 1 — what to think about</summary>

Two checks happen for every candidate, and the ORDER matters: is it redundant with what you
already picked, and does it fit what is left of the budget. Which one should run first, so a
redundant chunk is never charged against the budget at all? And when a candidate does not
fit, should the whole assembly process stop, or keep looking further down the ranking?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Keep a running list of selected chunks and a running token total, both starting empty. Walk
`ranked_chunks` in order. For each one: compute `_covered_fraction` of its own span against
the chunks already selected FROM THE SAME DOCUMENT; if that is `>= dedup_threshold`, skip it
and move to the next candidate. Otherwise compute its cost with `count_tokens`; if the
running total plus this cost would exceed `token_budget`, skip it (but keep scanning);
otherwise add it to the selected list and add its cost to the running total.
</details>

In [ ]:
def _covered_fraction(start: int, end: int, doc_id: int, spans: Sequence[Chunk]) -> float:
    """Given, not graded: the fraction of [start, end) already covered by `spans` from `doc_id`."""
    length = end - start
    if length <= 0:
        return 0.0
    same_doc = sorted((c.start, c.end) for c in spans if c.doc_id == doc_id)
    covered = 0
    cursor = start
    for s, e in same_doc:
        s, e = max(s, start), min(e, end)
        if e <= s:
            continue
        s = max(s, cursor)
        if e > s:
            covered += e - s
            cursor = max(cursor, e)
    return covered / length


def assemble_context(ranked_chunks: Sequence[Chunk], token_budget: int,
                      count_tokens: Callable[[str], int] = estimate_tokens,
                      dedup_threshold: float = DEDUP_OVERLAP_THRESHOLD) -> list[Chunk]:
    """Greedily assemble `ranked_chunks` (best first) into a context costing <= `token_budget`.

    Requirements, each of which is graded:
      * GREEDY BY SCORE, continuing the scan: a candidate that does not fit the remaining
        budget is SKIPPED, never a reason to stop — a smaller, later candidate may still fit.
      * a candidate is DROPPED (never added, never charged) if `_covered_fraction` of its own
        span against the chunks ALREADY selected from the SAME document is `>=
        dedup_threshold` — checked BEFORE the budget, so a redundant chunk never spends any
        of it.
      * the running total is the sum of `count_tokens(chunk.text)` over chunks actually added;
        a chunk is added only if doing so would not push that total over `token_budget`.
      * the result is in the order chunks were ADDED (their rank order), not sorted by
        position in the document.

    Example:
        >>> a = Chunk(0, 0, "x" * 10, 0, 10)
        >>> b = Chunk(0, 1, "y" * 10, 20, 30)
        >>> [c.chunk_id for c in assemble_context([a, b], token_budget=1000)]
        [0, 1]
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_assemble_context() -> None:
    cheap = Chunk(0, 0, "x" * 10, 0, 10)      # ~3 tokens at 3.5 chars/token
    costly = Chunk(0, 1, "y" * 100, 100, 200)  # ~29 tokens
    tiny = Chunk(0, 2, "z" * 10, 300, 310)     # ~3 tokens

    skip_then_fit = assemble_context([costly, tiny], token_budget=estimate_tokens("y" * 100) - 1)
    assert [c.chunk_id for c in skip_then_fit] == [2], (
        f"got chunk_ids {[c.chunk_id for c in skip_then_fit]} — a candidate that does not fit "
        "must be SKIPPED, not a reason to stop scanning the rest of the ranking"
    )

    dup = Chunk(0, 3, "x" * 6, 2, 8)   # its whole span, 2-8, lies inside `cheap` (0-10)
    deduped = assemble_context([cheap, dup], token_budget=1000)
    assert [c.chunk_id for c in deduped] == [0], (
        f"got chunk_ids {[c.chunk_id for c in deduped]} — chunk 3's span is entirely inside "
        "chunk 0's, already selected from the SAME document: it must be dropped, not added"
    )
    # A dropped chunk must cost nothing. This budget holds `cheap` and `tiny` with one token
    # to spare, so charging `dup` on its way to being dropped leaves no room for `tiny`.
    room = estimate_tokens(cheap.text) + estimate_tokens(tiny.text) + 1
    starved = assemble_context([cheap, dup, tiny], token_budget=room)
    assert [c.chunk_id for c in starved] == [0, 2], (
        f"got chunk_ids {[c.chunk_id for c in starved]} — chunk 3 was redundant, so it must "
        "never have been charged against the budget: check overlap BEFORE cost"
    )

    other_doc = Chunk(1, 0, "x" * 6, 2, 8)   # same numeric span, DIFFERENT document
    cross_doc = assemble_context([cheap, other_doc], token_budget=1000)
    assert len(cross_doc) == 2, (
        f"got {len(cross_doc)} chunk(s) — a chunk from a DIFFERENT document must never be "
        "deduplicated against this one, however much its numeric span overlaps"
    )

    order = assemble_context([tiny, cheap], token_budget=1000)
    assert [c.chunk_id for c in order] == [2, 0], (
        "the result must be in the order chunks were ADDED (their rank order), not sorted by "
        "position in the document"
    )
    print("exercise 4 looks right — assemble_context skips instead of stopping, drops a "
          "redundant chunk before spending budget on it, and never dedups across documents")

In [ ]:
_try("exercise 4", _check_assemble_context)

## 8. Exercise 5 — the metric that matters: `evidence_coverage`

`recall_at_k` (section 2) answers "was a chunk touching the evidence retrieved at all". This
is a DIFFERENT, stricter question: does the UNION of the chunks actually assembled into the
context fully contain the gold evidence span, with no gap — because a model reading the
assembled context sees exactly what made it in, nothing about what a retriever merely
considered.

<details><summary>💡 Hint 1 — what to think about</summary>

A chunk that overlaps the evidence span is not the same thing as the evidence span being
COVERED. Two chunks, neither containing the whole span, can still cover it TOGETHER if their
union has no gap across it. What is the smallest piece of information you need to decide
"covered", and does it come from `assembled` alone or does `target_doc_id` matter too?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Filter `assembled` down to chunks from `target_doc_id` only, and sort their `(start, end)`
spans by `start`. Walk them with a `cursor` starting at `evidence_start`: a span that starts
AFTER the cursor leaves a gap nothing earlier in the (sorted) list could have filled, so stop
and report "not covered". A span that starts at or before the cursor extends it to that
span's `end`, if that is further than the cursor already reached. Coverage holds the moment
the cursor reaches (or passes) `evidence_end`.
</details>

In [ ]:
def evidence_coverage(assembled: Sequence[Chunk], target_doc_id: int,
                       evidence_start: int, evidence_end: int) -> bool:
    """Whether `assembled`'s chunks from `target_doc_id` fully cover [evidence_start, evidence_end).

    Requirements, each of which is graded:
      * TRUE only when EVERY character offset in `[evidence_start, evidence_end)` is covered
        by the UNION of assembled chunks from `target_doc_id` — a chunk that merely OVERLAPS
        the evidence span, without the union covering all of it, is NOT coverage.
      * chunks from a DIFFERENT `doc_id` never count, however much their own numeric span
        happens to overlap.
      * a GAP between two assembled chunks that falls inside the evidence span must fail
        coverage, even when both chunks individually touch the span.
      * `evidence_end <= evidence_start` (an empty or malformed span) is vacuously covered:
        return `True`.

    Example:
        >>> chunks = [Chunk(0, 0, "...", 0, 10), Chunk(0, 1, "...", 10, 25)]
        >>> evidence_coverage(chunks, 0, 5, 20)
        True
        >>> evidence_coverage(chunks[:1], 0, 5, 20)
        False
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_evidence_coverage() -> None:
    whole = [Chunk(0, 0, "...", 0, 100)]
    assert evidence_coverage(whole, 0, 20, 80) is True, (
        "a single chunk that fully contains the evidence span must count as covered"
    )

    half = [Chunk(0, 0, "...", 0, 50)]
    assert evidence_coverage(half, 0, 20, 80) is False, (
        "a chunk that OVERLAPS the evidence span without containing all of it is NOT "
        "coverage — this is the difference between a retrieval hit and evidence coverage"
    )

    two_pieces = [Chunk(0, 0, "...", 0, 50), Chunk(0, 1, "...", 50, 100)]
    assert evidence_coverage(two_pieces, 0, 20, 80) is True, (
        "TWO assembled chunks whose union covers the whole span must count as covered, even "
        "though neither one alone does"
    )

    with_gap = [Chunk(0, 0, "...", 0, 30), Chunk(0, 1, "...", 60, 100)]
    assert evidence_coverage(with_gap, 0, 20, 80) is False, (
        "a GAP between two assembled chunks, inside the evidence span, must fail coverage even "
        "though both chunks individually overlap the span"
    )

    wrong_doc = [Chunk(1, 0, "...", 0, 100)]
    assert evidence_coverage(wrong_doc, 0, 20, 80) is False, (
        "a chunk from a DIFFERENT document must never count, however much its own numeric "
        "span overlaps — check doc_id, not just the offsets"
    )

    assert evidence_coverage([], 0, 5, 5) is True, (
        "an empty (evidence_end <= evidence_start) span is vacuously covered"
    )
    print("exercise 5 looks right — evidence_coverage requires the FULL span covered, by chunks "
          "from the right document, gaps and all, not merely 'some chunk overlapped it'")

In [ ]:
_try("exercise 5", _check_evidence_coverage)

In [ ]:
def _show_hit_without_coverage() -> None:
    docs = build_documents()
    queries = build_queries(docs)
    all_chunks: list[Chunk] = []
    for d in docs:
        all_chunks.extend(chunk_document(d, fixed_size_chunk, chunk_size=20, overlap=0))
    index = build_index([c.text for c in all_chunks])
    for q in queries:
        relevant = {i for i, c in enumerate(all_chunks)
                    if c.doc_id == q.doc_id and c.start < q.evidence_end and c.end > q.evidence_start}
        ranked_ids = [i for i, _ in bm25_rank(q.tokens, index)[:CANDIDATE_N]]
        hit = recall_at_k(ranked_ids, relevant, CANDIDATE_N) > 0.0
        assembled = assemble_context([all_chunks[i] for i in ranked_ids], TOKEN_BUDGET)
        cov = evidence_coverage(assembled, q.doc_id, q.evidence_start, q.evidence_end)
        if hit and not cov:
            touching = [c for c in assembled if c.doc_id == q.doc_id
                        and c.start < q.evidence_end and c.end > q.evidence_start]
            print(f"query {q.qid!r}: retrieval HIT (a chunk touching the evidence was "
                  f"retrieved) but evidence_coverage is FALSE")
            for chunk in touching or [None]:
                print("  assembled chunk:  " + (repr(chunk.text) if chunk else
                                                "(none — the budget left it out)"))
            print(f"  gold evidence:    {docs[q.doc_id].text[q.evidence_start:q.evidence_end]!r}")
            return
    print("no hit-without-coverage example found at chunk_size=20 — see section 9's sweep")


_try("hit without coverage demo", _show_hit_without_coverage,
     needs=("exercise 1", "exercise 4", "exercise 5"))

## 9. The scorecard: chunk size and overlap, against a measured coverage/cost curve

Every number below is computed once, here, by the functions above — recall@k (a retrieval
HIT: was a chunk touching the evidence retrieved) beside evidence_coverage (did the FULL
answer actually reach the assembled context), the tokens spent getting there, and how many
chunks each assembled context holds. When that last column sits near one, coverage is decided
by where one chunk's boundaries fall, not by the budget. Bootstrap intervals resample
DOCUMENTS, not queries: a document's two queries can land on the very same chunk, and then
they are one observation counted twice, which would print an interval narrower than the data
supports. Read the intervals, not just the means.

In [ ]:
def evaluate_chunking_policy(docs: Sequence[Document], queries: Sequence[RagQuery],
                              chunk_fn: Callable[..., list[ChunkSpan]], chunk_kwargs: dict,
                              token_budget: int = TOKEN_BUDGET,
                              candidate_n: int = CANDIDATE_N) -> dict:
    """Given, not graded: chunk every document, retrieve+assemble for every query, measure."""
    all_chunks: list[Chunk] = []
    for d in docs:
        all_chunks.extend(chunk_document(d, chunk_fn, **chunk_kwargs))
    index = build_index([c.text for c in all_chunks])
    hits, coverages, tokens_spent, n_assembled = [], [], [], []
    for q in queries:
        relevant = {i for i, c in enumerate(all_chunks)
                    if c.doc_id == q.doc_id and c.start < q.evidence_end and c.end > q.evidence_start}
        ranked_ids = [i for i, _ in bm25_rank(q.tokens, index)[:candidate_n]]
        # "hit": was ANY chunk touching the evidence retrieved at all — recall_at_k > 0 iff the
        # top-k intersects relevant at least once, whatever fraction of a multi-chunk evidence
        # span that turns out to be. This is the retrieval question; evidence_coverage below is
        # the different, stricter one: did the FULL span reach the assembled context.
        hits.append(1.0 if relevant and recall_at_k(ranked_ids, relevant, candidate_n) > 0.0
                    else 0.0)
        assembled = assemble_context([all_chunks[i] for i in ranked_ids], token_budget)
        coverages.append(float(evidence_coverage(assembled, q.doc_id, q.evidence_start,
                                                  q.evidence_end)))
        tokens_spent.append(sum(estimate_tokens(c.text) for c in assembled))
        n_assembled.append(len(assembled))
    # One value per DOCUMENT (the mean of its queries' coverage): the unit the bootstrap
    # resamples, because a document's queries share its chunks and are not independent.
    doc_coverages = [float(np.mean([cov for q, cov in zip(queries, coverages)
                                    if q.doc_id == d.doc_id])) for d in docs]
    return dict(hit=float(np.mean(hits)), coverage_ci=bootstrap_ci(doc_coverages, seed=SEED),
                tokens=float(np.mean(tokens_spent)), n_chunks=len(all_chunks),
                chunks_per_query=float(np.mean(n_assembled)),
                coverages=coverages, doc_coverages=doc_coverages)


def run_policy_sweep() -> None:
    docs = build_documents()
    queries = build_queries(docs)
    print(f"{len(docs)} documents, {len(queries)} queries, token_budget={TOKEN_BUDGET}\n")

    print("-- (a) chunk size: coverage against cost (fixed-size, overlap=0) --")
    configs = [("fixed", dict(chunk_size=cs, overlap=0)) for cs in (15, 20, 30, 45, 60, 90)]
    for name, kwargs in configs:
        r = evaluate_chunking_policy(docs, queries, fixed_size_chunk, kwargs)
        mean, lo, hi = r["coverage_ci"]
        print(f"  chunk_size={kwargs['chunk_size']:3d}  hit={r['hit']:.2f}  "
              f"coverage={mean:.2f} [{lo:.2f},{hi:.2f}]  tokens/query={r['tokens']:5.1f}  "
              f"chunks/query={r['chunks_per_query']:.1f}  n_chunks={r['n_chunks']}")

    print("\n-- (b) overlap: what it buys, and what it costs (chunk_size=30) --")
    baseline = None
    for ov in (0, 7, 15):
        r = evaluate_chunking_policy(docs, queries, fixed_size_chunk,
                                     dict(chunk_size=30, overlap=ov))
        mean, lo, hi = r["coverage_ci"]
        print(f"  overlap={ov:2d}  hit={r['hit']:.2f}  coverage={mean:.2f} [{lo:.2f},{hi:.2f}]  "
              f"tokens/query={r['tokens']:5.1f}  chunks/query={r['chunks_per_query']:.1f}")
        if ov == 0:
            baseline = r
        elif ov == 7:
            diff, dlo, dhi = paired_bootstrap_diff(baseline["doc_coverages"], r["doc_coverages"],
                                                   seed=SEED)
            print(f"    overlap 0 -> 7: paired coverage difference {diff:+.2f} "
                  f"[{dlo:+.2f},{dhi:+.2f}], for {r['tokens'] - baseline['tokens']:+.1f} more "
                  "tokens/query")

    print("\n-- (c) structure-aware chunking, and the policy this lesson picks --")
    best = None
    for mt in (30, 60, 90, 120):
        r = evaluate_chunking_policy(docs, queries, structure_chunk, dict(max_tokens=mt))
        mean, lo, hi = r["coverage_ci"]
        print(f"  structure max_tokens={mt:3d}  hit={r['hit']:.2f}  "
              f"coverage={mean:.2f} [{lo:.2f},{hi:.2f}]  tokens/query={r['tokens']:5.1f}  "
              f"chunks/query={r['chunks_per_query']:.1f}")
        # Policy rule, stated and applied, not just eyeballed: the cheapest configuration (of
        # every one swept, fixed-size or structure-aware) whose coverage MEAN is >= 0.95.
        if mean >= 0.95 and (best is None or r["tokens"] < best[1]["tokens"]):
            best = (f"structure_chunk(max_tokens={mt})", r)
    for name, kwargs in configs:
        r = evaluate_chunking_policy(docs, queries, fixed_size_chunk, kwargs)
        if r["coverage_ci"][0] >= 0.95 and (best is None or r["tokens"] < best[1]["tokens"]):
            best = (f"fixed_size_chunk(chunk_size={kwargs['chunk_size']})", r)

    print("\nPOLICY: the cheapest swept configuration reaching >= 95% mean evidence coverage.")
    if best is None:
        print("no configuration reached 95% coverage in this sweep — see the table above")
    else:
        name, r = best
        print(f"  chosen: {name}  ->  coverage={r['coverage_ci'][0]:.2f}, "
              f"{r['tokens']:.1f} tokens/query, {r['n_chunks']} chunks total")


_try("full policy sweep", run_policy_sweep,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 10. Common mistakes

- **Stopping `assemble_context` at the first chunk that does not fit.** A ranking's second
  candidate can be far cheaper than its first; a knapsack scan keeps looking.
- **Checking de-duplication AFTER spending the budget.** A redundant chunk must never be
  charged against `token_budget` at all — check overlap first, then cost.
- **De-duplicating across documents.** Two chunks from different documents can share the
  same numeric character range; only a same-document candidate can be redundant.
- **`evidence_coverage` returning true on ANY overlap.** A chunk that touches the evidence
  span without the union of assembled chunks containing all of it is not coverage — see
  exercise 5's `half` case. This is exactly the gap between `recall_at_k` (was something
  relevant retrieved) and this lesson's own metric (did the FULL answer arrive).
- **A heading that repeats a document's own unique identifier.** If it does, an "exact"
  query for that identifier can match the heading's chunk instead of the evidence's — this
  corpus's heading deliberately omits the case code for exactly this reason.
- **Cutting a paragraph mid-fact and not noticing.** `structure_chunk`'s whole reason to
  exist is that a paragraph boundary is a place a human author already chose not to split a
  thought; a fixed-size window has no such information.
- **Rounding `estimate_tokens` down, or letting it return 0 for non-empty text.** Both make
  the budget optimistic exactly where it matters — near the edge of what still fits.
- **Scoring a gold span in different units from the chunks.** A chunk spans first token to
  last token; a gold span that also counts a closing full stop can never be contained by a
  chunk that stops at the fact's last word. With `overlap=0`, the space or comma BETWEEN two
  fixed windows belongs to neither, so their union always has a gap; one token of overlap
  closes it.
- **Growing chunks without limit to keep the fact whole.** At a fixed budget, a chunk that
  alone costs more than the whole budget can never be assembled.

The last of these, measured rather than asserted:

In [ ]:
def _show_budget_collapse() -> None:
    docs = build_documents()
    d = docs[0]
    giant = chunk_document(d, fixed_size_chunk, chunk_size=1000, overlap=0)
    index = build_index([c.text for c in giant])
    ranked = bm25_rank(tokenize(f"case-{4800}"), index)
    assembled = assemble_context([giant[i] for i, _ in ranked], TOKEN_BUDGET)
    print(f"chunk_size=1000 (the whole document as ONE chunk, "
          f"{estimate_tokens(giant[0].text)} estimated tokens) against a budget of "
          f"{TOKEN_BUDGET}: {len(assembled)} chunk(s) assembled, "
          f"{sum(estimate_tokens(c.text) for c in assembled)} tokens actually spent")
    if estimate_tokens(giant[0].text) > TOKEN_BUDGET and not assembled:
        print("the one chunk costs more than the whole budget, so nothing was assembled at all.")
    else:
        print("this chunk still fits the budget — raise chunk_size or lower TOKEN_BUDGET to see "
              "the collapse.")


_try("budget collapse demo", _show_budget_collapse, needs=("exercise 1", "exercise 4"))

## 11. Self-check

1. `assemble_context` is scanning a ranking. The next candidate costs more than the budget
   that remains, but a cheaper candidate sits two places further down the same ranking. What
   should happen?
   - (a) skip this candidate and keep scanning; the cheaper one further down may still fit
   - (b) stop assembling — the ranking is exhausted for this query
   - (c) truncate the expensive candidate's text to make it fit

2. A chunk's span runs from character 100 to character 150. The query's gold evidence span
   runs from 120 to 200. That chunk is the ONLY one assembled for this query. What does
   `evidence_coverage` report?
   - (a) True — the chunk overlaps the evidence span
   - (b) True, because retrieval found a relevant chunk (recall@k would say so)
   - (c) False — the union of assembled spans does not contain the WHOLE evidence span

3. Suppose one row of section 9's table printed retrieval "hit" (recall) at 1.00 while its
   `evidence_coverage` sat far lower. What would that gap show?
   - (a) the retriever is broken and should be re-tuned
   - (b) a relevant chunk being retrieved is not the same fact as the model receiving the
     whole answer — assembly and chunk boundaries can still lose it
   - (c) `evidence_coverage` must have a bug, since recall already says "found"

4. Raising `chunk_size` without limit, at a FIXED token budget:
   - (a) can eventually LOWER coverage to zero, once a single chunk costs more than the
     whole budget and nothing is assembled at all
   - (b) always raises evidence coverage, since bigger chunks hold more text
   - (c) has no effect on coverage, only on how many chunks exist

5. `structure_chunk` merges a heading block into the chunk that follows it rather than
   leaving the heading to close a chunk by itself. Why?
   - (a) headings are not tokenised, so they would not count against the budget anyway
   - (b) `tools/notebooks.py` requires every chunk to start with a heading
   - (c) a heading alone carries none of a query's evidence, so a chunk that is JUST the
     heading spends budget without being able to help evidence coverage

Answers come with this lesson's worked solution when you enrol on Synapsa.

## What you built

`fixed_size_chunk` and `structure_chunk` turn one document into many; `estimate_tokens`
prices each one; `assemble_context` spends a fixed budget on the best-ranked, least
redundant of them; `evidence_coverage` is the only one of these five that looks at the
ANSWER, not the chunk — and section 9 measured, on this corpus, how often the other four
get it wrong without it.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_fixed_size_chunk, ()),
        ("exercise 2", _check_structure_chunk, ()),
        ("exercise 3", _check_estimate_tokens, ()),
        ("exercise 4", _check_assemble_context, ()),
        ("exercise 5", _check_evidence_coverage, ()),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends the run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.